In [6]:
!pip install lightgbm xgboost catboost

  Obtaining dependency information for lightgbm from https://files.pythonhosted.org/packages/5e/23/f8b28ca248bb629b9e08f877dd2965d1994e1674a03d67cd10c5246da248/lightgbm-4.6.0-py3-none-win_amd64.whl.metadata
   ---------------------------------------- 1.5/1.5 MB 5.8 MB/s eta 0:00:00


In [8]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("1. Orijinal veriler yükleniyor...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

# ==========================================
# 2. ÜLKE İSİMLERİNİ NORMALİZE ETME
# ==========================================
# (İngilizce/Türkçe karmaşasını çözüyoruz)
ulke_map = {
    'Netherlands': 'Hollanda', 'Germany': 'Almanya', 'France': 'Fransa', 
    'Spain': 'Ispanya', 'Italy': 'Italya', 'UK': 'Ingiltere', 
    'United Kingdom': 'Ingiltere', 'USA': 'ABD', 'United States': 'ABD', 
    'China': 'Cin', 'South Korea': 'Guney Kore', 'Japan': 'Japonya',
    'Turkey': 'Turkiye'
}
train['ulke'] = train['ulke'].replace(ulke_map)
test['ulke'] = test['ulke'].replace(ulke_map)

# ==========================================
# 3. EKSİK VERİLERİ (NaN) DOLDURMA
# ==========================================
# Sızıntı (Leakage) olmaması için test setindeki eksikleri de TRAIN'in istatistikleriyle dolduruyoruz.
num_cols_with_nan = ['vucut_kitle_indeksi', 'uyku_oncesi_kafein_mg', 'stres_skoru']
cat_cols_with_nan = ['meslek', 'kronotip', 'ruh_sagligi_durumu']

for col in num_cols_with_nan:
    median_val = train[col].median()
    train[col].fillna(median_val, inplace=True)
    test[col].fillna(median_val, inplace=True)

for col in cat_cols_with_nan:
    mode_val = train[col].mode()[0]
    train[col].fillna(mode_val, inplace=True)
    test[col].fillna(mode_val, inplace=True)

# ==========================================
# 4. ÖZELLİK MÜHENDİSLİĞİ (FEATURE ENGINEERING)
# ==========================================
# Daha önce eklediğin ve faydalı olan kolonları tekrar üretiyoruz.
def add_features(df):
    df = df.copy()
    
    # 1. uyku_kalitesi (rem + derin)
    df['uyku_kalitesi'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    
    # 2. rem_oran
    df['rem_oran'] = df['rem_yuzdesi'] / (df['uyku_kalitesi'] + 1e-5) # 0'a bölünme hatasını engellemek için +1e-5
    
    # 3. stres_x_calisma
    df['stres_x_calisma'] = df['stres_skoru'] * df['gunluk_calisma_saati']
    
    # 4. yas_x_stres
    df['yas_x_stres'] = df['yas'] * df['stres_skoru']
    
    # 5. dijital_yuk (Ekran süresi ve kafein etkileşimi)
    df['dijital_yuk'] = df['uyku_oncesi_ekran_suresi_dk'] * (df['uyku_oncesi_kafein_mg'] + 1)
    
    # 6. aktif_skor (Günlük adım ve nabız)
    df['aktif_skor'] = df['gunluk_adim_sayisi'] / (df['dinlenik_nabiz_bpm'] + 1)
    
    # 7. uyku_baskisi (Uykuya dalma süresi ve uyanma)
    df['uyku_baskisi'] = df['uykuya_dalma_suresi_dk'] * (df['gecelik_uyanma_sayisi'] + 1)
    
    # 8. stres_x_ruh (Ruh sağlığını sıralı sayısala çevirip stresle çarpma)
    ruh_map = {'Saglikli': 1, 'Notr': 2, 'Anksiyete ve depresyon': 3, 'Depresyon': 4}
    ruh_encoded = df['ruh_sagligi_durumu'].map(ruh_map).fillna(2) # Eşleşmeyen kalırsa 'Notr' say
    df['stres_x_ruh'] = df['stres_skoru'] * ruh_encoded
    
    return df

print("4. Yeni özellikler (Feature Engineering) ekleniyor...")
train = add_features(train)
test = add_features(test)

# ==========================================
# 5. TEMİZLENMİŞ DOSYALARI KAYDETME
# ==========================================
train.to_csv('train_clean.csv', index=False)
test.to_csv('test_clean.csv', index=False)

print("\n[+] HARİKA! Veri temizleme tamamlandı.")
print(f"Oluşan Train Boyutu: {train.shape}")
print(f"Oluşan Test Boyutu: {test.shape}")
print("Artık 'train_clean.csv' ve 'test_clean.csv' dosyaların hazır.")

1. Orijinal veriler yükleniyor...
4. Yeni özellikler (Feature Engineering) ekleniyor...

[+] HARİKA! Veri temizleme tamamlandı.
Oluşan Train Boyutu: (56000, 32)
Oluşan Test Boyutu: (24000, 31)
Artık 'train_clean.csv' ve 'test_clean.csv' dosyaların hazır.


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. VERİ YÜKLEME VE HAZIRLIK
# ==========================================
print("Veriler yükleniyor...")
train = pd.read_csv('train_clean.csv')
test = pd.read_csv('test_clean.csv')

# ID'leri ve hedef değişkeni ayıralım
train_ids = train['id']
test_ids = test['id']
y = train['bilissel_performans_skoru']

# Özellikler (Features)
X = train.drop(columns=['id', 'bilissel_performans_skoru'])
X_test = test.drop(columns=['id'])

# Kategorik sütunları otomatik bul
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# ==========================================
# 2. VERİ ÖN İŞLEME (MODELLERE GÖRE AYIRMA)
# ==========================================
# A) CATBOOST İÇİN: Orijinal string formatında bırakıyoruz
X_cat = X.copy()
X_test_cat = X_test.copy()
for col in cat_cols:
    X_cat[col] = X_cat[col].astype(str)
    X_test_cat[col] = X_test_cat[col].astype(str)

# B) LIGHTGBM & XGBOOST İÇİN: Target Encoding + Label Encoding
X_lgb_xgb = X.copy()
X_test_lgb_xgb = X_test.copy()

# Sadece hedef potansiyeli yüksek kolonlara Target Encoding
target_encode_cols = [c for c in ['meslek', 'ruh_sagligi_durumu'] if c in cat_cols]
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for col in target_encode_cols:
    X_lgb_xgb[col + '_target_enc'] = np.nan
    for train_idx, val_idx in kf.split(X_lgb_xgb):
        X_tr, X_val = X_lgb_xgb.iloc[train_idx], X_lgb_xgb.iloc[val_idx]
        target_mean = y.iloc[train_idx].groupby(X_tr[col]).mean()
        X_lgb_xgb.loc[val_idx, col + '_target_enc'] = X_val[col].map(target_mean)
    
    # NaN kalanları genel ortalama ile doldur
    global_mean = y.mean()
    X_lgb_xgb[col + '_target_enc'].fillna(global_mean, inplace=True)
    
    # Test seti için encoding
    test_target_mean = y.groupby(X_lgb_xgb[col]).mean()
    X_test_lgb_xgb[col + '_target_enc'] = X_test_lgb_xgb[col].map(test_target_mean).fillna(global_mean)

# Geriye kalan kategorik kolonları (ve orijinalleri) Label Encode yapalım
for col in cat_cols:
    le = LabelEncoder()
    # Bilinmeyen kategorileri engellemek için Train ve Test'i birleştirip fitliyoruz
    full_data = pd.concat([X_lgb_xgb[col], X_test_lgb_xgb[col]], axis=0).astype(str)
    le.fit(full_data)
    X_lgb_xgb[col] = le.transform(X_lgb_xgb[col].astype(str))
    X_test_lgb_xgb[col] = le.transform(X_test_lgb_xgb[col].astype(str))

# ==========================================
# 3. K-FOLD EĞİTİMİ VE OOF (OUT-OF-FOLD) TAHMİNLERİ
# ==========================================
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

test_preds_lgb = np.zeros(len(X_test))
test_preds_xgb = np.zeros(len(X_test))
test_preds_cat = np.zeros(len(X_test))

print("\n10-Fold Çapraz Doğrulama Eğitimleri Başlıyor...\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"--- FOLD {fold+1} Eğitiliyor ---")
    
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    X_tr_lgb_xgb, X_val_lgb_xgb = X_lgb_xgb.iloc[train_idx], X_lgb_xgb.iloc[val_idx]
    X_tr_cat, X_val_cat = X_cat.iloc[train_idx], X_cat.iloc[val_idx]
    
    # 3.1. LightGBM
    try:
        callbacks_lgb = [lgb.early_stopping(stopping_rounds=100, verbose=False)]
    except:
        callbacks_lgb = None # Eski versiyon LightGBM kullanıyorsan hata vermesin diye
        
    model_lgb = LGBMRegressor(n_estimators=1000, learning_rate=0.03, random_state=42)
    model_lgb.fit(X_tr_lgb_xgb, y_tr, eval_set=[(X_val_lgb_xgb, y_val)], 
                  eval_metric='rmse', callbacks=callbacks_lgb if callbacks_lgb else None)
    oof_lgb[val_idx] = model_lgb.predict(X_val_lgb_xgb)
    test_preds_lgb += model_lgb.predict(X_test_lgb_xgb) / kf.n_splits
    
    # 3.2. XGBoost
    model_xgb = XGBRegressor(n_estimators=1000, learning_rate=0.03, random_state=42, early_stopping_rounds=100)
    model_xgb.fit(X_tr_lgb_xgb, y_tr, eval_set=[(X_val_lgb_xgb, y_val)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict(X_val_lgb_xgb)
    test_preds_xgb += model_xgb.predict(X_test_lgb_xgb) / kf.n_splits
    
    # 3.3. CatBoost (Tuning sonrası elde ettiğimiz iyi parametrelerle)
    model_cat = CatBoostRegressor(iterations=900, learning_rate=0.035, l2_leaf_reg=2.0, 
                                  eval_metric='RMSE', random_seed=42, cat_features=cat_cols)
    model_cat.fit(X_tr_cat, y_tr, eval_set=[(X_val_cat, y_val)], early_stopping_rounds=100, verbose=False)
    oof_cat[val_idx] = model_cat.predict(X_val_cat)
    test_preds_cat += model_cat.predict(X_test_cat) / kf.n_splits

print(f"\n--- TEKİL MODEL OOF SKORLARI ---")
print(f"LightGBM RMSE : {np.sqrt(mean_squared_error(y, oof_lgb)):.5f}")
print(f"XGBoost RMSE  : {np.sqrt(mean_squared_error(y, oof_xgb)):.5f}")
print(f"CatBoost RMSE : {np.sqrt(mean_squared_error(y, oof_cat)):.5f}")

# ==========================================
# 4. SCIPY İLE OPTİMİZE EDİLMİŞ AĞIRLIKLARI BULMA
# ==========================================
def rmse_blend(weights):
    blend_pred = (weights[0] * oof_lgb) + (weights[1] * oof_xgb) + (weights[2] * oof_cat)
    return np.sqrt(mean_squared_error(y, blend_pred))

# Ağırlıkların kısıtlamaları (Toplamları 1 olacak ve hepsi 0-1 arasında olacak)
init_weights = [0.33, 0.33, 0.34]
bounds = ((0, 1), (0, 1), (0, 1))
cons = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})

res = minimize(rmse_blend, init_weights, method='SLSQP', bounds=bounds, constraints=cons)

print("\n--- OPTİMİZASYON SONUÇLARI ---")
print(f"Bulunan Ağırlıklar : LGB={res.x[0]:.4f}, XGB={res.x[1]:.4f}, CAT={res.x[2]:.4f}")
print(f"Mükemmel Blend OOF RMSE : {res.fun:.5f}")

# ==========================================
# 5. FİNAL DOSYASINI HAZIRLAMA (SUBMISSION)
# ==========================================
final_test_preds = (res.x[0] * test_preds_lgb) + (res.x[1] * test_preds_xgb) + (res.x[2] * test_preds_cat)

submission = pd.DataFrame({
    'id': test_ids,
    'bilissel_performans_skoru': final_test_preds
})
submission.to_csv('submission_v5_weighted_blend.csv', index=False)
print("\n[+] Başarılı! 'submission_v5_weighted_blend.csv' hazır. Yükleyebilirsin!")

Veriler yükleniyor...

10-Fold Çapraz Doğrulama Eğitimleri Başlıyor...

--- FOLD 1 Eğitiliyor ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003713 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4849
[LightGBM] [Info] Number of data points in the train set: 50400, number of used features: 32
[LightGBM] [Info] Start training from score 5.916820
--- FOLD 2 Eğitiliyor ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004562 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4845
[LightGBM] [Info] Number of data points in the train set: 50400, number of used features: 32
[LightGBM] [Info] Start training from score 5.909929
--- FOLD 3 Eğitiliyor ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003876 seconds.
You can set `force_col_wise=true` to remove the overhead.
[Lig